In [1]:
from sage.libs.ecl import ecl_eval
ecl_eval("(ext:set-limit 'ext:heap-size 0)")   # 0 = no ECL-side limit

<ECL: 0>

In [2]:
var('n e_t u_bar v_bar f_4t f_5t g_4t g_5t Xi dt du dv gamma F_phi e_phi y K G m')
var('u_bar m G eta gamma e_t n Xi')
var('y y_0 h TE y_1PN y_15PN y_2PN y_25PN Xi chi_A chi_S delta omega')
var('e_t gamma y_inv e0 omega_dash A B flagTail e0 x0 flagHF SO chi w0 f0 j f_var f0_var e y0 kappaA kappaS horiz z')

(e_t,
 gamma,
 y_inv,
 e0,
 omega_dash,
 A,
 B,
 flagTail,
 e0,
 x0,
 flagHF,
 SO,
 chi,
 w0,
 f0,
 j,
 f_var,
 f0_var,
 e,
 y0,
 kappaA,
 kappaS,
 horiz,
 z)

In [3]:
filename = "NLO_e0/et_e0_NLO_3PN.txt"

with open(filename, "r") as f:
    content = f.read()

# Extract the part after "dt_du_final ="
import re
match = re.search(r"et_e0_NLO_3PN\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    # Convert string back into a Sage expression
    et_e0_NLO_3PN = (sage_eval(expr_str, locals=globals())).subs({SO^2:0},{SO^3:0})

In [4]:
import re
filename = "../x_e_files/x_3PN.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"x_3PN\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    xe_expr = (((sage_eval(expr_str, locals=globals()))*(e_t^(12/19)/e0^(12/19))).expand()).subs({SO^2:0},{SO^3:0})

In [5]:
progress_file = "NNLO_e0/progress.txt"
count = 0

def update_progress(step_name=""):
    global count
    count += 1
    with open(progress_file, "w") as f:
        f.write("# Expansion of count\n")
        f.write("# Plain ASCII output from Sage\n\n")
        f.write(f"count = {count}\n")
        if step_name:
            f.write(f"step = {step_name}\n")

# Fast version (truncated power series) — `_faster` notebook

This notebook is `et_e0_5.ipynb` plus a fast implementation of every order.

* **N, 1PN, 1.5PN, 2PN**: your original cells are kept unchanged. After each order a
  **FAST** cell recomputes the same order and prints whether it is **exactly** equal to your result.
* **2.5PN, 3PN**: only the fast version (your symbolic version takes weeks).
* Fast results are written to `NNLO_e0_faster/` (your cells still write `NNLO_e0/`).

**How the fast version works** (no change of physics, only of how the algebra is done):

1. Everything is put in one polynomial ring over exact rationals `QQ` with variables `e0, Xi` and every
   other symbol as a coefficient variable: `chi^(1/18)`, `x0^(1/2)`, `eta`, spins, flags, `pi`,
   `euler_gamma`, `log(2)`, `log(3)`, `log(5)`, `log(x0)`, `log(chi)`, `log(e0)`.
2. After **every** multiplication, terms are dropped (`trunc`). At order `Xi^k` we keep
   `e0` power ≤ 5 and `Xi` power ≤ k: exactly the `Xi^k e0^0..5` terms your
   `.taylor((e0,0),(Xi,0),5+k)` produces, without ever building `Xi^(k+1)...` or the full expression.
3. `A.taylor((e_t,0),(e0,0),4).subs({e_t: S})` is done term by term:
   `coef*e0^a*e_t^b -> coef*e0^(a+b)*u^b` with `u = S/e0`, and `u^b` = truncated binomial series.
   `log(e_t) -> log(e0) + log(u)` with `log(u)` a truncated log series.
4. `(...)^(19/12)` is a truncated binomial series.
5. **Dummy pieces** (`.subs({e_t:y})`, `.subs({e0:y})`): they are multiplied by `Xi^(k-1)` at order
   `Xi^k`, and no input has a `Xi^1` term (checked by an `assert`), so they can never reach the saved
   `Xi^k` coefficient. They are left out (= 0); your own cells show the same thing.

In [ ]:
# =====================================================================================
#  Truncated power-series machinery: fast, exact replacement for symbolic .taylor
# =====================================================================================
import operator
from sage.symbolic.operators import add_vararg, mul_vararg
ADD = (operator.add, add_vararg)
MUL = (operator.mul, mul_vararg)
SPECIAL = (e0, e_t, Xi)          # handled separately, never coefficient atoms
def is_log(op):
    return hasattr(op, "name") and op.name() == "log"

# ---- 1) find every symbol / constant appearing in the inputs -------------------------
def collect_atoms(exprs):
    """roots[s] = d  means symbol s only appears as powers s^(k/d)  (e.g. chi -> 18, x0 -> 2).
    consts      = non-symbol atoms like pi, euler_gamma, log(2), log(x0), log(chi)."""
    roots, consts = {}, {}
    def walk(ex):
        if ex.is_numeric():
            return
        op = ex.operator()
        if op is None:                                   # symbol or constant (pi, ...)
            if ex.is_symbol():
                if ex not in SPECIAL:
                    roots.setdefault(ex, 1)
            else:
                consts[str(ex)] = ex
            return
        if op in ADD or op in MUL:
            for a in ex.operands():
                walk(a)
            return
        if op is operator.pow:
            base, ee = ex.operands()
            ee = QQ(ee)
            if base.is_symbol() and base not in SPECIAL:
                roots[base] = lcm(roots.get(base, 1), ee.denominator())
            else:
                walk(base)
            return
        if is_log(op):
            arg = ex.operands()[0]
            if arg not in (e0, e_t):                     # log(e0), log(e_t) handled separately
                consts[str(ex)] = ex
            return
        raise ValueError("unsupported function in %s" % ex)
    for ex in exprs:
        walk(SR(ex).expand())
    return roots, consts

def _name(s):
    return "".join(ch if ch.isalnum() else "_" for ch in s) + "_r"

def build_ring(exprs):
    """Flat Laurent polynomial ring over QQ in  e0, Xi  and all coefficient atoms."""
    global L, e0_s, Xi_s, ROOTS, ATOM_GEN, GEN_SR, LOGE0
    ROOTS, consts = collect_atoms(exprs)
    for s, d in ROOTS.items():                         # log(s) is needed for log(s^(k/d))
        if d > 1:
            consts.setdefault(str(log(s)), log(s))
    names = ["e0_s", "Xi_s"] + [_name(str(s)) for s in ROOTS] + [_name(k) for k in consts] + ["log_e0_r"]
    L = LaurentPolynomialRing(QQ, names)
    g = L.gens()
    e0_s, Xi_s = g[0], g[1]
    ATOM_GEN, GEN_SR = {}, [e0, Xi]
    i = 2
    for s, d in ROOTS.items():                         # generator = s^(1/d)
        ATOM_GEN[s] = g[i]; GEN_SR.append(s**(QQ(1)/d)); i += 1
    for k, ex in consts.items():
        ATOM_GEN[k] = g[i]; GEN_SR.append(ex); i += 1
    LOGE0 = g[i]; GEN_SR.append(log(e0))
    print("series ring variables:", names)

# ---- 2) SR <-> ring ----------------------------------------------------------------
def to_P(ex):
    """Exact conversion of a symbolic expression into the ring L."""
    ex = SR(ex)
    if ex.is_numeric():
        return L(QQ(ex))
    op, args = ex.operator(), ex.operands()
    if op is None:
        if ex == e0: return e0_s
        if ex == Xi: return Xi_s
        if ex.is_symbol(): return ATOM_GEN[ex]**ROOTS[ex]
        return ATOM_GEN[str(ex)]
    if op in ADD: return sum(to_P(a) for a in args)
    if op in MUL: return prod(to_P(a) for a in args)
    if op is operator.pow:
        base, ee = args[0], QQ(args[1])
        if base.is_symbol() and base in ROOTS:
            k = ee * ROOTS[base]
            assert k in ZZ, ex
            return ATOM_GEN[base]**ZZ(k)
        assert ee in ZZ, ex
        if ee >= 0:
            return to_P(base)**ZZ(ee)
        b = to_P(base)
        assert len(b.monomials()) == 1, ex                 # only invert a single monomial
        return b**ZZ(ee)
    if is_log(op):
        if args[0] == e0: return LOGE0
        return ATOM_GEN[str(ex)]
    raise ValueError("cannot convert %s" % ex)

def to_SR(f):
    """Ring L -> symbolic expression (chi^(1/18) etc. restored)."""
    out = SR(0)
    for m, q in f.dict().items():
        out += q * prod(GEN_SR[i]**m[i] for i in range(len(m)) if m[i])
    return out

# ---- 3) truncation -----------------------------------------------------------------
def keep(m, N):
    """N integer: total degree in (e0, Xi) <= N  (like .taylor((Xi,0),(e0,0),N)).
    N = (Ne0, NXi): e0 power <= Ne0 AND Xi power <= NXi (separate limits)."""
    if isinstance(N, tuple):
        return m[0] <= N[0] and m[1] <= N[1]
    return m[0] + m[1] <= N

def trunc(f, N):
    return L({m: q for m, q in f.dict().items() if keep(m, N)})

def xi_part(f, k):
    """The Xi^k part of f (Xi^k factor included), like f.coefficient(Xi,k)*Xi^k."""
    return L({m: q for m, q in f.dict().items() if m[1] == k})

# ---- 4) series of f^b and log(f) ---------------------------------------------------
def split_unit(f):
    """f = U * (1 + delta): U = (e0, Xi)-free part, must be a single monomial with coefficient 1."""
    U = L({m: q for m, q in f.dict().items() if m[0] == 0 and m[1] == 0})
    assert len(U.monomials()) == 1 and U.coefficients() == [1], U
    return U, f * U**-1 - 1

def _order_len(N):
    return sum(N) if isinstance(N, tuple) else N

def series_pow(f, b, N):
    """f^b truncated at order N:  U^b * sum_n binom(b, n) delta^n."""
    b = QQ(b)
    U, delta = split_unit(f)
    e = U.exponents()[0]
    assert all((x * b) in ZZ for x in e), (U, b)
    Ub = L({tuple(ZZ(x * b) for x in e): 1})
    res, term = L(1), L(1)
    for n in range(1, _order_len(N) + 1):
        term = trunc(term * delta, N) * (b - n + 1) / n
        if term == 0:
            break
        res += term
    return trunc(Ub * res, N)

def log_of_unit(U):
    """log of a monomial in the root atoms, e.g. log(chi^(-19/18)) = -19/18 * log(chi)."""
    e = U.exponents()[0]
    out = L(0)
    for s, d in ROOTS.items():
        k = e[list(L.gens()).index(ATOM_GEN[s])]
        if k:
            out += QQ(k) / d * ATOM_GEN[str(log(s))]
    return out

def series_log(f, N):
    """log(f) truncated at order N:  log(U) + sum_n (-1)^(n+1) delta^n / n."""
    U, delta = split_unit(f)
    res, term = log_of_unit(U), L(1)
    for n in range(1, _order_len(N) + 1):
        term = trunc(term * delta, N)
        if term == 0:
            break
        res += (-1)**(n + 1) * term / n
    return res

# ---- 5) the replacement for  A.taylor((e_t,0),(e0,0),tord).subs({e_t: et_sub}) ---------
def monomial_split(t):
    """t = coef * e0^a * e_t^b * log(e0)^p * log(e_t)^q  ->  (coef, a, b, p, q)."""
    a = b = QQ(0); p = q = 0
    coef = SR(1)
    facs = t.operands() if t.operator() in MUL else [t]
    for fct in facs:
        base, ee = (fct.operands()[0], fct.operands()[1]) if fct.operator() is operator.pow else (fct, 1)
        if base == e0:            a += QQ(ee)
        elif base == e_t:         b += QQ(ee)
        elif base == log(e0):     p += ZZ(ee)
        elif base == log(e_t):    q += ZZ(ee)
        else:                     coef *= fct
    return coef, a, b, p, q

def xe_series(A, et_sub, tord, N):
    """Series version of  A.taylor((e_t,0),(e0,0),tord).subs({e_t: et_sub})  at order N.
    Each term coef*e0^a*e_t^b (a+b integer) becomes coef*e0^(a+b)*u^b with u = e_t/e0,
    log(e_t) becomes log(e0) + log(u)."""
    assert all(m[0] >= 1 for m in et_sub.exponents()), "e_t must be divisible by e0"
    u = et_sub * e0_s**-1
    cache, out = {}, L(0)
    logu = None
    terms = SR(A).expand()
    terms = terms.operands() if terms.operator() in ADD else [terms]
    for t in terms:
        coef, a, b, p, q = monomial_split(t)
        if a + b > tord:                       # same cut as .taylor((e_t,0),(e0,0),tord)
            continue
        assert (a + b) in ZZ and a + b >= 0, (t, a, b)
        if b not in cache:
            cache[b] = series_pow(u, b, N)
        piece = to_P(coef) * e0_s**ZZ(a + b) * cache[b]
        if p:
            piece *= LOGE0**p
        if q:
            if logu is None:
                logu = series_log(u, N)
            piece = trunc(piece * (LOGE0 + logu)**q, N)
        out += trunc(piece, N)
    return out

In [ ]:
# ---- FAST: setup (ring, inputs, helpers) ----
import os, time
os.makedirs("NNLO_e0_faster", exist_ok=True)
build_ring([xe_expr, et_e0_NLO_3PN, chi])

# A_f[k] = xe_expr.coefficient(Xi,k).coefficient(x0^(1+k/2))  -- same pieces as in your cells
A_f = {k: xe_expr.coefficient(Xi, k).coefficient(x0**(1 + QQ(k)/2)) for k in (0, 2, 3, 4, 5, 6)}
# E_f[k] = et_e0_NLO_3PN.coefficient(Xi,k)*Xi^k  (expanded: its e_t terms at Xi^4, Xi^6 cancel)
E_f = {k: to_P((et_e0_NLO_3PN.coefficient(Xi, k) * Xi**k).expand()) for k in range(7)}
# the dummy-y pieces can only matter through a Xi^1 term; there is none:
assert xe_expr.coefficient(Xi, 1) == 0 and et_e0_NLO_3PN.coefficient(Xi, 1) == 0
pref_f = to_P(e0 / chi**(QQ(19)/18))                       # e0/chi^(19/18)
x0h_f = {k: to_P(x0**(QQ(k)/2)) for k in range(7)}          # x0^(k/2)

def stage(pieces, k):
    """One PN order. pieces = [(j, A_j, e_t substitution), ...] means
         X = sum_j  Xi^j * x0^(j/2) * A_j.taylor((e_t,0),(e0,0),4).subs({e_t: substitution})
       returns the Xi^k part of  X^(19/12) * e0/chi^(19/18),  with e0 up to e0^6."""
    O = (5, k)
    X = L(0)
    for j, Aj, sub in pieces:
        xe_j = xe_series(Aj, sub, 4, (5, k - j))           # times Xi^j below, so Xi <= k-j suffices
        X += trunc(xe_j * Xi_s**j * x0h_f[j], O)
    return xi_part(series_pow(X, 19/12, O) * pref_f, k)

def compare(name, orig_expr, k, fast, fname):
    """difference = (your original Xi^k part) - (fast Xi^k part), computed exactly.
    Printed to the log and saved to NNLO_e0_faster/diff_et_e0_NNLO_<fname>.txt. Must be 0."""
    diff = to_P(orig_expr.expand().coefficient(Xi, k) * Xi**k) - fast
    with open("NNLO_e0_faster/diff_et_e0_NNLO_%s.txt" % fname, "w") as fh:
        fh.write("# difference (original - fast) of the Xi^%d part at %s; 0 means identical\n\n" % (k, name))
        fh.write("diff_et_e0_NNLO_%s =\n" % fname)
        fh.write(str(to_SR(diff)))
        fh.write("\n\n")
    if diff == 0:
        print("%-6s difference (original - fast) = 0   -> IDENTICAL" % name, flush=True)
    else:
        print("!!!!!! %s difference (original - fast) is NOT zero: %d terms, see NNLO_e0_faster/diff_et_e0_NNLO_%s.txt"
              % (name, len(diff.monomials()), fname), flush=True)
        print("difference =", to_SR(diff), flush=True)
    return diff == 0

def save_fast(name, f):
    with open("NNLO_e0_faster/et_e0_NNLO_%s.txt" % name, "w") as fh:
        fh.write("# Expansion of et_e0_NNLO_%s (fast power-series version)\n" % name)
        fh.write("# Plain ASCII output from Sage\n\n")
        fh.write("et_e0_NNLO_%s =\n" % name)
        fh.write(str(to_SR(f)))
        fh.write("\n\n")

**N**

In [6]:
xe_N=xe_expr.coefficient(Xi,0).coefficient(x0).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_NLO_3PN.coefficient(Xi,0)})

In [7]:
et_e0_N=(((((((xe_N))^(19/12))).taylor(e0,0,5))))*(e0/chi^(19/18))

In [8]:
filename = "NNLO_e0/et_e0_NNLO_N.txt"


with open(filename, "w") as f:
    f.write("# Expansion of et_e0_NNLO_N\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("et_e0_NNLO_N =\n")
    f.write(str(et_e0_N.coefficient(Xi,0)))
    f.write("\n\n")

In [ ]:
# ---- FAST N  (= xe_N, et_e0_N above) ----
t_f = time.time()
etN_f = stage([(0, A_f[0], E_f[0])], 0)
print("N      fast time: %.2f s" % (time.time() - t_f))
compare("N", et_e0_N, 0, etN_f, "N")
save_fast("N", etN_f)

**1 PN**

In [9]:
xe_1PN_expr=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_NLO_3PN.coefficient(Xi,2)*Xi^2})


In [10]:
xe_1PN_1=xe_1PN_expr*Xi^2*x0+xe_N
et_e0_1PN=((((xe_1PN_1)^(19/12))).taylor((Xi,0),(e0,0),7))*(e0/chi^(19/18))


In [11]:
filename = "NNLO_e0/et_e0_NNLO_1PN.txt"


with open(filename, "w") as f:
    f.write("# Expansion of et_e0_NNLO_1PN\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("et_e0_NNLO_1PN =\n")
    f.write(str(et_e0_1PN.coefficient(Xi,2)*Xi^2))
    f.write("\n\n")

In [ ]:
# ---- FAST 1PN  (= xe_N, xe_1PN_expr, xe_1PN_1, et_e0_1PN above) ----
# note: like your xe_N, the Newtonian piece here uses e_t = NLO Newtonian only
t_f = time.time()
et1_f = stage([(0, A_f[0], E_f[0]),                       # xe_N
               (2, A_f[2], etN_f + E_f[2])], 2)           # xe_1PN_expr
print("1PN    fast time: %.2f s" % (time.time() - t_f))
compare("1PN", et_e0_1PN, 2, et1_f, "1PN")
save_fast("1PN", et1_f)

In [19]:
show(et_e0_1PN.expand())

4411380221/808455168*Xi^6*chi^(17/18)*e0*eta^3*x0^3 - 2851724129/269485056*Xi^6*chi^(5/18)*e0*eta^3*x0^3 - 63438782569/7545581568*Xi^6*chi^(17/18)*e0*eta^2*x0^3 + 1292068037/269485056*Xi^6*e0*eta^3*x0^3/chi^(7/18) + 41009819581/2515193856*Xi^6*chi^(5/18)*e0*eta^2*x0^3 + 912294776741/211276283904*Xi^6*chi^(17/18)*e0*eta*x0^3 + 267588055/808455168*Xi^6*e0*eta^3*x0^3/chi^(19/18) - 18580856593/2515193856*Xi^6*e0*eta^2*x0^3/chi^(7/18) - 2192359219/359313408*Xi^4*chi^(5/18)*e0^3*eta^2*x0^2 - 589750349609/70425427968*Xi^6*chi^(5/18)*e0*eta*x0^3 - 13119447220849/17747207847936*Xi^6*chi^(17/18)*e0*x0^3 - 549729485/1077940224*Xi^6*e0*eta^2*x0^3/chi^(19/18) + 1460196505/179656704*Xi^4*e0^3*eta^2*x0^2/chi^(7/18) + 267205922477/70425427968*Xi^6*e0*eta*x0^3/chi^(7/18) + 31527683591/5030387712*Xi^4*chi^(5/18)*e0^3*eta*x0^2 + 8481029139301/5915735949312*Xi^6*chi^(5/18)*e0*x0^3 + 3185814065/359313408*Xi^4*e0^3*eta^2*x0^2/chi^(19/18) + 7905500665/30182326272*Xi^6*e0*eta*x0^3/chi^(19/18) - 327001429595/32697520128*Xi^4*e0^3*eta*x0^2/chi^(7/18) - 3842611057753/5915735949312*Xi^6*e0*x0^3/chi^(7/18) - 453390495499/281701711872*Xi^4*chi^(5/18)*e0^3*x0^2 - 659753/196992*Xi^4*chi^(5/18)*e0*eta^2*x0^2 - 3338788505/359313408*Xi^4*e0^3*eta^2*x0^2/chi^(11/6) - 106691166739/9342148608*Xi^4*e0^3*eta*x0^2/chi^(19/18) - 113686717685/2535315406848*Xi^6*e0*x0^3/chi^(19/18) + 3150863507/479084544*Xi^2*e0^5*eta*x0/chi^(7/18) + 5479341746545/1831061127168*Xi^4*e0^3*x0^2/chi^(7/18) + 194045/98496*Xi^4*e0*eta^2*x0^2/chi^(7/18) + 3258933767/179656704*Xi^4*e0^3*eta^2*x0^2/chi^(5/2) + 9487717/2757888*Xi^4*chi^(5/18)*e0*eta*x0^2 + 170426008381/65395040256*Xi^4*e0^3*eta*x0^2/chi^(11/6) - 11147601665/479084544*Xi^2*e0^5*eta*x0/chi^(19/18) + 1845026319707/523160322048*Xi^4*e0^3*x0^2/chi^(19/18) + 271663/196992*Xi^4*e0*eta^2*x0^2/chi^(19/18) - 7092926885/359313408*Xi^4*e0^3*eta^2*x0^2/chi^(19/6) - 45311656423/13414367232*Xi^2*e0^5*x0/chi^(7/18) - 2790505/1378944*Xi^4*e0*eta*x0^2/chi^(7/18) - 252731074573/32697520128*Xi^4*e0^3*eta*x0^2/chi^(5/2) - 136440113/154441728*Xi^4*chi^(5/18)*e0*x0^2 + 4074514822747/3662122254336*Xi^4*e0^3*x0^2/chi^(11/6) + 3061519891285/174386774016*Xi^2*e0^5*x0/chi^(19/18) - 558101/393984*Xi^4*e0*eta*x0^2/chi^(19/18) + 102001329265/5030387712*Xi^4*e0^3*eta*x0^2/chi^(19/6) + 654631/131328*Xi^2*e0^3*eta*x0/chi^(7/18) + 40129445/77220864*Xi^4*e0*x0^2/chi^(7/18) + 575447233/79847424*Xi^2*e0^5*eta*x0/chi^(5/2) - 213231090871/261580161024*Xi^4*e0^3*x0^2/chi^(5/2) + 14528137028993/13144163549184*e0^7/chi^(19/18) - 1482433/131328*Xi^2*e0^3*eta*x0/chi^(19/18) + 8025889/22063104*Xi^4*e0*x0^2/chi^(19/18) + 3092267495/79847424*Xi^2*e0^5*eta*x0/chi^(19/6) - 1466851603085/281701711872*Xi^4*e0^3*x0^2/chi^(19/6) - 9414059/3677184*Xi^2*e0^3*x0/chi^(7/18) - 472055540597/29064462336*Xi^2*e0^5*x0/chi^(5/2) + 386822573/47803392*Xi^2*e0^3*x0/chi^(19/18) - 699589093187/29064462336*Xi^2*e0^5*x0/chi^(19/6) + 197/72*Xi^2*e0*eta*x0/chi^(7/18) + 173171/131328*Xi^2*e0^3*eta*x0/chi^(5/2) - 4108380839/479084544*Xi^2*e0^5*eta*x0/chi^(83/18) + 15994231/6653952*e0^5/chi^(19/18) - 197/72*Xi^2*e0*eta*x0/chi^(19/18) - 89629527689279/4381387849728*e0^7/chi^(19/6) + 654631/131328*Xi^2*e0^3*eta*x0/chi^(19/6) - 9901169371/479084544*Xi^2*e0^5*eta*x0/chi^(95/18) - 2833/2016*Xi^2*e0*x0/chi^(7/18) - 142057039/47803392*Xi^2*e0^3*x0/chi^(5/2) + 2706383369971/174386774016*Xi^2*e0^5*x0/chi^(83/18) + 2833/2016*Xi^2*e0*x0/chi^(19/18) - 9414059/3677184*Xi^2*e0^3*x0/chi^(19/6) + 142385851919/13414367232*Xi^2*e0^5*x0/chi^(95/18) + 3323/1824*e0^3/chi^(19/18) - 11042329/1108992*e0^5/chi^(19/6) + 140969652369941/4381387849728*e0^7/chi^(95/18) + e0/chi^(19/18) - 3323/1824*e0^3/chi^(19/6) + 50259743/6653952*e0^5/chi^(95/18) - 168548511070979/13144163549184*e0^7/chi^(133/18)

**1.5 PN**

I basically test if we can ignore substituting PN accurate e_t expressions into orders that are not going to contribute. And yes, you can ignore it here. expressions with _real do not ignore contributions from useless orders

In [15]:
xe_15_N=xe_expr.coefficient(Xi,0).coefficient(x0).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_NLO_3PN.coefficient(Xi,3)*Xi^3})

In [16]:
xe_15_1PN=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:y})
#xe_15_1PN_real=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).subs({e_t:et_e0_N+et_e0_1PN*Xi^2})

In [17]:
xe_15PN=xe_expr.coefficient(Xi,3).coefficient(x0^(5/2)).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N})

In [ ]:
xe_15PN_2=xe_15_N+Xi^2*xe_15_1PN*x0+Xi^3*xe_15PN*x0^(3/2)
#xe_15PN_2_real=xe_15_N+Xi^2*xe_15_1PN_real*x0+Xi^3*xe_15PN*x0^(3/2)

et_e0_15PN=(((xe_15PN_2)^(19/12)).taylor((e0,0),(Xi,0),8))*(e0/chi^(19/18))
#et_e0_15PN_real=((xe_15PN_2_real)^(19/12)).taylor((e0,0),(Xi,0),6)*(e0/chi^(19/18))



In [ ]:
filename = "NNLO_e0/et_e0_NNLO_15PN.txt"


with open(filename, "w") as f:
    f.write("# Expansion of et_e0_NNLO_15PN\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("et_e0_NNLO_15PN =\n")
    f.write(str(et_e0_15PN.coefficient(Xi,3)*Xi^3))
    f.write("\n\n")

In [ ]:
# ---- FAST 1.5PN  (= xe_15_N, xe_15PN, et_e0_15PN above; xe_15_1PN is the dummy-y piece) ----
t_f = time.time()
et15_f = stage([(0, A_f[0], etN_f + et1_f + E_f[3]),      # xe_15_N
                (3, A_f[3], etN_f)], 3)                   # xe_15PN
print("1.5PN  fast time: %.2f s" % (time.time() - t_f))
compare("1.5PN", et_e0_15PN, 3, et15_f, "15PN")
save_fast("15PN", et15_f)

**2 PN**

In [ ]:
xe_2_N=xe_expr.coefficient(Xi,0).coefficient(x0).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_15PN.coefficient(Xi,3)*Xi^3+et_e0_NLO_3PN.coefficient(Xi,4)*Xi^4})
#To save computational time, newtonian order not important 

In [ ]:
xe_2_1PN=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2})
#xe_2_1PN_real=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).subs({e_t:et_e0_NLO_3PN.coefficient(Xi,0)+et_e0_NLO_3PN.coefficient(Xi,2)*Xi^2})

In [ ]:
xe_2_15PN=xe_expr.coefficient(Xi,3).coefficient(x0^(5/2)).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e0:y})
#xe_2_15PN_real=xe_expr.coefficient(Xi,3).coefficient(x0^(5/2)).taylor((e_t,0),(e0,0),0).subs({e0:et_e0_1PN})
#To save computational time, 1.5PN order not important 

In [ ]:
xe_2PN=xe_expr.coefficient(Xi,4).coefficient(x0^3).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N})

In [ ]:
xe_2PN_1=xe_2PN*Xi^4*x0^2+xe_2_15PN*Xi^3*x0^(3/2)+xe_2_1PN*Xi^2*x0+xe_2_N
#xe_2PN_1_real=xe_2PN*Xi^4*x0^2+xe_2_15PN*Xi^3*x0^(3/2)+xe_2_1PN_real*Xi^2*x0+xe_2_N


et_e0_2PN=(((xe_2PN_1^(19/12))).taylor((e0,0),(Xi,0),9))*(e0/chi^(19/18))
#et_e0_2PN_real=(((xe_2PN_1_real^(19/12))).taylor((e0,0),(Xi,0),7))*(e0/chi^(19/18))

In [ ]:
filename = "NNLO_e0/et_e0_NNLO_2PN.txt"


with open(filename, "w") as f:
    f.write("# Expansion of et_e0_NNLO_2PN\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("et_e0_NNLO_2PN =\n")
    f.write(str(et_e0_2PN.coefficient(Xi,4)*Xi^4))
    f.write("\n\n")

In [ ]:
#show(et_e0_2PN.coefficient(Xi,4).expand().coefficient(e0,3)-et_e0_2PN_real.coefficient(Xi,4).expand().coefficient(e0,3))


In [ ]:
# ---- FAST 2PN  (= xe_2_N, xe_2_1PN, xe_2PN, et_e0_2PN above; xe_2_15PN is the dummy-y piece) ----
t_f = time.time()
et2_f = stage([(0, A_f[0], etN_f + et1_f + et15_f + E_f[4]),   # xe_2_N
               (2, A_f[2], etN_f + et1_f),                     # xe_2_1PN
               (4, A_f[4], etN_f)], 4)                         # xe_2PN
print("2PN    fast time: %.2f s" % (time.time() - t_f))
compare("2PN", et_e0_2PN, 4, et2_f, "2PN")
save_fast("2PN", et2_f)

**2.5 PN** (fast version only)

In [ ]:
# ---- FAST 2.5PN  (same pieces as your xe_25_N, xe_25_1PN, xe_25_15PN, xe_25PN; xe_25_2PN is dummy-y) ----
t_f = time.time()
et25_f = stage([(0, A_f[0], etN_f + et1_f + et15_f + et2_f + E_f[5]),   # xe_25_N
                (2, A_f[2], etN_f + et1_f + et15_f),                    # xe_25_1PN
                (3, A_f[3], etN_f + et1_f + et15_f + et2_f),            # xe_25_15PN
                (5, A_f[5], etN_f)], 5)                                 # xe_25PN
print("2.5PN  fast time: %.2f s" % (time.time() - t_f))
save_fast("25PN", et25_f)
update_progress("fast 2.5PN done")

**3 PN** (fast version only)

In [ ]:
# ---- FAST 3PN  (same pieces as your xe_3_N, xe_3_1PN, xe_3_15PN, xe_3_2PN, xe_3PN; xe_3_25PN is dummy-y) ----
t_f = time.time()
et3_f = stage([(0, A_f[0], etN_f + et1_f + et15_f + et2_f + et25_f + E_f[6]),  # xe_3_N
               (2, A_f[2], etN_f + et1_f + et15_f + et2_f),                    # xe_3_1PN
               (3, A_f[3], etN_f + et1_f + et15_f),                            # xe_3_15PN
               (4, A_f[4], etN_f + et1_f),                                     # xe_3_2PN
               (6, A_f[6], etN_f)], 6)                                         # xe_3PN
print("3PN    fast time: %.2f s" % (time.time() - t_f))
save_fast("3PN_only", et3_f)          # the Xi^6 part alone
update_progress("fast 3PN done")

In [ ]:
# ---- FAST total, same as your et_e0_NNLO_3PN = N + 1PN Xi^2 + ... + 3PN Xi^6 ----
et_e0_NNLO_3PN_fast = to_SR(etN_f + et1_f + et15_f + et2_f + et25_f + et3_f)
save_fast("3PN", etN_f + et1_f + et15_f + et2_f + et25_f + et3_f)
print("files written to NNLO_e0_faster/:")
for fn in sorted(os.listdir("NNLO_e0_faster")):
    print("   ", fn)
